In [1]:
!pip install neurokit2 xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 10.8 MB/s eta 0:00:00


In [2]:
# ==========================================
# CELL 1: Imports and Config
# ==========================================
import os, pickle, warnings, json, time
import numpy as np, pandas as pd
from scipy.signal import welch
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, cohen_kappa_score
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk

warnings.filterwarnings('ignore'); np.random.seed(42); tf.random.set_seed(42)

DATA_PATH = '/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE = '/kaggle/working'; os.makedirs(SAVE, exist_ok=True)
SUBJECT_IDS = [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES = ['relaxed','mild','moderate','high']; NCLS = 4

# Live Inference Optimization
WINDOW = 60  # Reduced for faster causal response
STEP = 5
EWMA_HALFLIVES = {'fast':60, 'medium':300, 'slow':1800}
POPULATION_RR_MS = 780.0        
ROLL_WINDOW = 20                

print("TF", tf.__version__, "GPU", len(tf.config.list_physical_devices('GPU'))>0)

TF 2.20.0 GPU True


In [3]:
# ==========================================
# CELL 2: Preprocessing (Matching notebook-improvements_2.ipynb exactly)
# ==========================================
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f:
        data = pickle.load(f, encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg = nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs); rp = info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr, ts):
    rr = rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)] = np.nan
    for i in range(1, len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1] > 0.20: rr[i] = np.nan
    m = np.isnan(rr)
    if m.any(): rr[m] = np.interp(np.where(m)[0], np.where(~m)[0], rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap = np.interp(rp/fe, np.arange(len(wt))/ft, wt)
    return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out = []
    for i in range(len(rp)-1):
        seg = labels[rp[i]:rp[i+1]]; v = seg[seg > 0]
        out.append(0 if len(v) == 0 else np.bincount(v).argmax())
    return np.array(out)

# Causal Replacements for Live Deployment
def ewma_causal(x, halflife):
    a = 1 - np.exp(np.log(0.5)/max(halflife, 1))
    o = np.empty(len(x), dtype=float)
    state = float(POPULATION_RR_MS) if len(x)==0 else float(POPULATION_RR_MS)
    for i in range(len(x)):
        state = a*x[i] + (1-a)*state
        o[i] = state
    return o

def causal_zscore(x, halflife=300):
    a = 1 - np.exp(np.log(0.5)/max(halflife, 1))
    mu = np.empty(len(x)); sd = np.empty(len(x))
    m = float(x[0]) if len(x) else 0.0; v = 1.0
    for i in range(len(x)):
        d = x[i] - m
        m = m + a*d
        v = (1-a)*(v + a*d*d)
        mu[i] = m; sd[i] = np.sqrt(max(v, 1e-8))
    return (x-mu)/(sd+1e-8)

def roll_rmssd_causal(x, w=ROLL_WINDOW):
    o = np.zeros(len(x))
    for i in range(len(x)):
        seg = x[max(0, i-w+1):i+1]
        o[i] = np.sqrt(np.mean(np.diff(seg)**2)) if len(seg)>1 else 0.0
    return o

def roll_sdnn_causal(x, w=ROLL_WINDOW):
    o = np.zeros(len(x))
    for i in range(len(x)):
        seg = x[max(0, i-w+1):i+1]
        o[i] = np.std(seg) if len(seg)>1 else 0.0
    return o

In [4]:
# ==========================================
# CELL 3: Feature Building
# ==========================================
def hrv_features(w, fs=4.0):
    rr, diff = np.array(w), np.diff(w)
    mean_rr = np.mean(rr); sdnn = np.std(rr); rmssd = np.sqrt(np.mean(diff**2))
    pnn50 = np.sum(np.abs(diff)>50)/len(diff)*100; cv = sdnn/mean_rr
    t = np.cumsum(rr)/1000.0; u = np.interp(np.arange(0, t[-1], 1/fs), t, rr)
    fr, psd = welch(u, fs=fs, nperseg=min(256, len(u)))
    vlf = TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf = TRAPZ(psd[(fr>=0.04)&(fr<0.15)]); hf = TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf = lf/(hf+1e-8); lf_nu = lf/(lf+hf+1e-8)
    sd1 = np.sqrt(0.5)*np.std(diff); sd2 = np.sqrt(max(2*sdnn**2 - 0.5*np.var(diff),0))
    sdr = sd1/(sd2+1e-8)
    return np.array([mean_rr, sdnn, rmssd, pnn50, cv, vlf, lf, hf, lf_hf, lf_nu, sd1, sd2, sdr])

def resid_features(rw):
    r = np.array(rw)
    return np.array([np.mean(r), np.std(r), np.max(np.abs(r)), np.polyfit(np.arange(len(r)), r, 1)[0], np.sum(r**2)/len(r)])

def circ_features(ts):
    t, hour = ts%86400, (ts%86400)/3600.0
    cort = 0.6*np.exp(-0.5*((hour-8)/1.5)**2) + 0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400), np.cos(2*np.pi*t/86400), np.sin(2*np.pi*t/5400), np.cos(2*np.pi*t/5400), cort])

def circ7(ts):
    t = ts%86400; hour = t/3600.0
    return np.array([np.sin(2*np.pi*t/86400), np.cos(2*np.pi*t/86400), np.sin(2*np.pi*t/5400), np.cos(2*np.pi*t/5400),
        0.6*np.exp(-0.5*((hour-8)/1.5)**2) + 0.3*np.exp(-0.5*((hour-15)/1.5)**2),
        np.sin(2*np.pi*(hour-23)/24), np.cos(2*np.pi*(hour-23)/24)])

X_seq, X_circ, X_xgb, y_all, groups = [], [], [], [], []
for sid in SUBJECT_IDS:
    try:
        chest, wt, labels = load_subject(sid); ecg = chest['ECG'].flatten()
        rr, ts, rp = extract_rr_from_ecg(ecg); temp = align_temp(wt, rp); rr, ts = clean_rr(rr, ts)
        rl = labels_to_rr(labels, rp); keep = rl > 0
        rrk, tk, tsk, lk = rr[keep], temp[keep], ts[keep], rl[keep]
        
        # RMSSD-tertile intensity proxy (from Notebook_Improvements_2.ipynb)
        new = np.zeros(len(lk), dtype=int); si = np.where(lk == 2)[0]
        if len(si) > 0:
            srr = rrk[si]; loc = []
            for i in range(len(srr)):
                w = srr[max(0, i-15):i+15]; dd = np.diff(w)
                loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc = np.array(loc); p33, p66 = np.percentile(loc, 33), np.percentile(loc, 66)
            for i, idx in enumerate(si): new[idx] = (1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        
        # Apply Causal Tracking (from Notebook-CausalRetrain.ipynb)
        base = {k: ewma_causal(rrk, hl) for k, hl in EWMA_HALFLIVES.items()}
        res_med = rrk - base['medium']
        tbase_med = ewma_causal(tk, EWMA_HALFLIVES['medium'])
        temp_res = tk - tbase_med

        rn = causal_zscore(rrk)
        rm = roll_rmssd_causal(rn)
        sd = roll_sdnn_causal(rn)
        hr = 60000 / (rrk+1e-8)
        rrn = causal_zscore(res_med)
        tn = causal_zscore(tk)
        trn = causal_zscore(temp_res)

        for s in range(0, len(rrk)-WINDOW, STEP):
            e = s+WINDOW; mid = s+WINDOW//2; bi = min(mid, len(tsk)-1)
            seq = np.stack([rn[s:e], rm[s:e], sd[s:e], hr[s:e], rrn[s:e], tn[s:e], trn[s:e]], axis=-1)
            try:
                xgbf = np.concatenate([
                    hrv_features(rrk[s:e]), resid_features(res_med[s:e]),
                    np.array([base['fast'][e-1], base['slow'][e-1]]), circ_features(tsk[bi])])
            except Exception: continue
            
            X_seq.append(seq); X_circ.append(circ7(tsk[bi])); X_xgb.append(xgbf)
            y_all.append(new[mid]); groups.append(int(sid))
            
    except Exception as e: print(f"S{sid} FAIL {e}")

X_seq, X_circ = np.array(X_seq, np.float32), np.array(X_circ, np.float32)
X_xgb, y_all, groups = np.array(X_xgb), np.array(y_all, np.int32), np.array(groups, np.int32)
print("seq", X_seq.shape, "xgb", X_xgb.shape, "classes", np.bincount(y_all))

seq (12026, 60, 7) xgb (12026, 25) classes [8774 1101 1080 1071]


In [5]:
# ==========================================
# CELL 4: Novel MS-CGCA Architecture & Loss
# ==========================================
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def call(self, yt, yp):
        yt = tf.cast(yt, tf.int32)
        ce = tf.keras.losses.sparse_categorical_crossentropy(yt, yp)
        pt = tf.reduce_sum(tf.one_hot(yt, 4) * yp, axis=-1)
        return tf.pow(1.0 - pt, self.gamma) * ce

def build_novel_ms_cgca(window=WINDOW, nch=7, ncirc=7, ncls=4):
    si = layers.Input(shape=(window, nch), name='sequence')
    ci = layers.Input(shape=(ncirc,), name='circadian')

    # 1. Multi-Scale Causal Convolutions (Dilated to strictly prevent future leakage)
    conv_1 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=1)(si)
    conv_2 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=2)(si)
    conv_4 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=4)(si)
    
    x = layers.Concatenate()([conv_1, conv_2, conv_4])
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    # 2. Causal Sequence Processing (Unidirectional LSTM)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.Dropout(0.4)(x)

    # 3. Circadian-Guided Cross-Attention
    # Transforms circadian context into a search query over the sequence
    c_proj = layers.Dense(128, activation='relu')(ci)
    c_proj = layers.RepeatVector(window // 2)(c_proj) 
    
    attn = layers.Attention()([c_proj, x])  # Query = Circadian, Key/Value = Sequence
    x = layers.GlobalAveragePooling1D()(attn)
    
    # 4. Classification Head
    c_flat = layers.Dense(32, activation='relu')(ci)
    merged = layers.Concatenate()([x, c_flat])
    
    out = layers.Dense(64, activation='relu')(merged)
    out = layers.Dropout(0.4)(out)
    out = layers.Dense(ncls, activation='softmax')(out)

    return Model([si, ci], out, name='MS_CGCA_Net')

def freeze_extractor(m):
    for l in m.layers:
        ln = l.__class__.__name__.lower()
        l.trainable = not any(k in ln for k in ['conv', 'lstm', 'batchnorm', 'attention', 'pooling'])
    return m

def make_xgb():
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, objective='multi:softprob', num_class=4, eval_metric='mlogloss', 
        random_state=42, n_jobs=-1)

print("Novel MS-CGCA architecture defined.")

Novel MS-CGCA architecture defined.


In [6]:
# ==========================================
# CELL 5: 3-Way Ensemble LOSO-CV Loop
# ==========================================
def strat_calib(sub_idx, y, frac=0.2, minpc=5):
    lab = y[sub_idx]; ntot = int(len(sub_idx)*frac); calib = []
    rng = np.random.RandomState(42)
    for c in np.unique(lab):
        pos = sub_idx[lab==c]; take = min(max(minpc, ntot//len(np.unique(lab))), len(pos))
        calib.extend(rng.choice(pos, take, replace=False))
    calib = np.array(sorted(calib)); ev = np.array([i for i in sub_idx if i not in set(calib)])
    return calib, ev

logo = LeaveOneGroupOut()
fold_store = []

print("Initiating 3-Way Ensemble Training (XGB + MS-CGCA + Finetuned)...")
for tr, te in logo.split(X_xgb, y_all, groups):
    s = int(np.unique(groups[te])[0])
    calib_local, eval_local = strat_calib(te, y_all)
    print(f"S{s:02d} | Calib: {len(calib_local):<4} | Eval: {len(eval_local):<4}", end=' | ', flush=True)

    # 1. XGBoost Base
    sc = StandardScaler(); Xtr = sc.fit_transform(X_xgb[tr]); Xev = sc.transform(X_xgb[eval_local])
    xgb = make_xgb()
    xgb.fit(Xtr, y_all[tr], sample_weight=compute_sample_weight('balanced', y_all[tr]), verbose=False)
    p_xgb = xgb.predict_proba(Xev)

    # 2. Population MS-CGCA
    cw = compute_class_weight('balanced', classes=np.unique(y_all[tr]), y=y_all[tr])
    pop = build_novel_ms_cgca(window=WINDOW)
    pop.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=SparseFocalLoss(2.0), metrics=['accuracy'])
    cb = [callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)]
    
    pop.fit([X_seq[tr], X_circ[tr]], y_all[tr], validation_split=0.15, epochs=100, batch_size=32,
            class_weight=dict(enumerate(cw)), callbacks=cb, verbose=0)
    p_cnn = pop.predict([X_seq[eval_local], X_circ[eval_local]], verbose=0)

    # 3. Personalised Fine-Tuned Head
    ft = build_novel_ms_cgca(window=WINDOW); ft.set_weights(pop.get_weights()); ft = freeze_extractor(ft)
    ft.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss=SparseFocalLoss(2.0), metrics=['accuracy'])
    
    cwdc = None
    if len(np.unique(y_all[calib_local])) >= 2:
        cwc = compute_class_weight('balanced', classes=np.unique(y_all[calib_local]), y=y_all[calib_local])
        cwdc = dict(zip(np.unique(y_all[calib_local]), cwc))
        
    ft.fit([X_seq[calib_local], X_circ[calib_local]], y_all[calib_local], epochs=15, batch_size=8, class_weight=cwdc, verbose=0)
    p_ft = ft.predict([X_seq[eval_local], X_circ[eval_local]], verbose=0)

    fold_store.append({'sub': s, 'y': y_all[eval_local].tolist(), 'p_xgb': p_xgb.tolist(), 
                       'p_cnn': p_cnn.tolist(), 'p_ft': p_ft.tolist()})
    print("Done")
    tf.keras.backend.clear_session()

Initiating 3-Way Ensemble Training (XGB + MS-CGCA + Finetuned)...
S02 | Calib: 140  | Eval: 567  | 

I0000 00:00:1786447279.211519      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786447279.217977      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Done
S03 | Calib: 124  | Eval: 500  | Done
S04 | Calib: 132  | Eval: 536  | Done
S05 | Calib: 144  | Eval: 583  | Done
S06 | Calib: 144  | Eval: 587  | Done
S07 | Calib: 140  | Eval: 574  | Done
S08 | Calib: 152  | Eval: 624  | Done
S09 | Calib: 160  | Eval: 650  | Done
S10 | Calib: 192  | Eval: 785  | Done
S11 | Calib: 184  | Eval: 742  | Done
S13 | Calib: 180  | Eval: 736  | Done
S14 | Calib: 184  | Eval: 742  | Done
S15 | Calib: 164  | Eval: 660  | Done
S16 | Calib: 176  | Eval: 709  | Done
S17 | Calib: 160  | Eval: 655  | Done


In [7]:
# ==========================================
# CELL 6: Nested Weight Selection & Eval
# ==========================================
def macro_f1(yt, yp, K=4):
    f = []
    for c in range(K):
        tp = np.sum((yp==c)&(yt==c)); fp = np.sum((yp==c)&(yt!=c)); fn = np.sum((yp!=c)&(yt==c))
        f.append(0.0 if tp==0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(f))

def blend(f, w):
    wf, wx, wc = w
    return wx * np.array(f['p_xgb']) + wc * np.array(f['p_cnn']) + wf * np.array(f['p_ft'])

GRID = [(wf, round(wx*(1-wf),4), round((1-wx)*(1-wf),4)) 
        for wf in [0.0, 0.1, 0.2, 0.3] for wx in np.arange(0.3, 0.71, 0.1)]

nested_preds, nested_y = [], []
for i, fS in enumerate(fold_store):
    inner = [f for j, f in enumerate(fold_store) if j != i]
    innerY = np.concatenate([f['y'] for f in inner])
    
    def inner_f1(w, inner=inner, innerY=innerY):
        preds = np.concatenate([blend(f, w).argmax(1) for f in inner])
        return macro_f1(innerY, preds)
    
    w_star = max(GRID, key=inner_f1)
    nested_preds.append(blend(fS, w_star).argmax(1))
    nested_y.append(fS['y'])

nested_y_arr = np.concatenate(nested_y)
nested_pred_arr = np.concatenate(nested_preds)

print("="*50)
print("FINAL CAUSAL MS-CGCA ENSEMBLE (60-BEAT)")
print("="*50)
print(f"Macro F1 : {macro_f1(nested_y_arr, nested_pred_arr):.4f}")
print(f"Kappa    : {cohen_kappa_score(nested_y_arr, nested_pred_arr, weights='quadratic'):.4f}")
print(f"Accuracy : {np.mean(nested_y_arr == nested_pred_arr):.4f}")
print("="*50)

FINAL CAUSAL MS-CGCA ENSEMBLE (60-BEAT)
Macro F1 : 0.6825
Kappa    : 0.8497
Accuracy : 0.9193
